# Tracking and player features
Reusable definitions only. No reference labels, downloading or training.

In [ ]:
from pathlib import Path
import hashlib
import json
import warnings
import xml.etree.ElementTree as ET

import numpy as np
from tqdm.auto import tqdm


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def input_blocks(data):
    """Return continuous model-frame intervals, resetting at half boundaries."""
    cuts = np.r_[0, np.flatnonzero(
        (np.diff(data['frame']) != 1) | (np.diff(data['period']) != 0)
    ) + 1, len(data['frame'])]
    return [(int(start), int(end)) for start, end in zip(cuts[:-1], cuts[1:])]


def read_tracking(raw_path, metadata_path, match):
    """Convert the original tactical ATD to the frozen 10 Hz input contract.

    ATD record IDs start at one. Output CSV frames start at zero. Track slots
    identify observations within a team, not persistent player identities.
    """
    if file_sha256(raw_path) != match['tracking_sha256']:
        raise ValueError('The tactical tracking file differs from the expected source.')
    if file_sha256(metadata_path) != match['metadata_sha256']:
        raise ValueError('The tactical metadata file differs from the expected source.')

    root = ET.parse(metadata_path).getroot()
    team_ids = [team.get('id') for team in root.iter('Team')]
    slot_teams = [player.get('teamId') for player in root.iter('Player')]
    if len(team_ids) != 2 or not slot_teams or not set(slot_teams).issubset(team_ids):
        raise ValueError('Expected two teams and an assigned team for every track slot.')
    teams = np.array([team_ids.index(team) for team in slot_teams], dtype=np.int8)
    team_slots = [np.flatnonzero(teams == team) for team in (0, 1)]
    max_slots = max(map(len, team_slots))

    # Official half boundaries are input timing metadata, not possession labels.
    step = match['video_fps'] / match['model_fps']
    grids, periods = [], []
    for period, first, last in match['halves']:
        grid = first + np.arange(int(np.floor((last - first) / step)) + 1) * step
        grids.extend(grid)
        periods.extend([period] * len(grid))
    grid_video = np.asarray(grids)
    source_video = np.floor(grid_video + 1e-8).astype(np.int32)
    period = np.asarray(periods, dtype=np.int8)
    lookup = {int(frame): index for index, frame in enumerate(source_video)}
    length = len(source_video)
    positions = np.zeros((length, 2, max_slots, 2), dtype=np.float32)
    observed = np.zeros(positions.shape[:-1], dtype=bool)
    ball = np.zeros((length, 2), dtype=np.float32)
    ball_observed = np.zeros(length, dtype=bool)
    pitch = np.array([match['pitch_metres'][0], -match['pitch_metres'][1]])

    count = 0
    with Path(raw_path).open() as stream:
        for video_frame, line in enumerate(tqdm(stream, total=match['frames'], desc='Read tactical tracking')):
            record, player_text, ball_text = line.strip().split(':')
            if int(record) != video_frame + 1:
                raise ValueError(f'ATD frame order mismatch at video frame {video_frame}.')
            count += 1
            index = lookup.get(video_frame)
            if index is None:
                continue
            players = np.fromstring(player_text.replace(';', ','), sep=',').reshape(-1, 3)[:, :2]
            raw_ball = np.fromstring(ball_text, sep=',')[:2]
            if len(players) != len(teams) or raw_ball.shape != (2,):
                raise ValueError(f'Unexpected ATD record shape at video frame {video_frame}.')
            for team, slots in enumerate(team_slots):
                xy = (players[slots] - [.5, .5]) * pitch
                valid = np.isfinite(xy).all(axis=1)
                xy[~valid] = 0
                positions[index, team, :len(slots)] = xy
                observed[index, team, :len(slots)] = valid
            if np.isfinite(raw_ball).all():
                ball[index] = (raw_ball - [.5, .5]) * pitch
                ball_observed[index] = True
    if count != match['frames']:
        raise ValueError(f'Expected {match["frames"]} ATD rows, found {count}.')

    # Hold the latest available model-grid prediction; never interpolate labels.
    mapping = np.full(match['frames'], -1, dtype=np.int32)
    for half, first, last in match['halves']:
        indices = np.flatnonzero(period == half)
        rows = np.arange(first, last + 1)
        mapping[rows] = indices[np.searchsorted(grid_video[indices], rows, side='right') - 1]
    valid = mapping >= 0
    assert np.all(source_video[mapping[valid]] <= np.flatnonzero(valid))
    return dict(positions=positions, observed=observed, ball=ball,
                ball_observed=ball_observed, frame=np.arange(length, dtype=np.int32),
                period=period, source_video=source_video, grid_video=grid_video,
                mapping_25hz=mapping)


def player_features(data):
    """Compute 140 player-only features with the original numerical conventions.

    Twenty per-frame features: 12 spatial quartiles, 2 observation counts and
    6 nearest-neighbour motion quartiles. Append past and future means over
    1, 3 and 10 seconds. Ball coordinates are not read by this function.
    """
    positions = data['positions'].copy()
    mask = data['observed']
    length = len(positions)
    positions[~mask] = np.nan
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        quartiles = np.nanquantile(positions, [.25, .5, .75], axis=2)
        quartiles = quartiles.transpose(1, 2, 0, 3).reshape(length, -1) / 50
    quartiles = np.nan_to_num(quartiles).astype(np.float32)
    counts = np.minimum(mask.sum(axis=2), 14) / 11
    motion = np.zeros((length, 6), dtype=np.float32)

    # Set distances avoid assuming that a slot follows one player across time.
    for low, high in input_blocks(data):
        for start in range(low + 10, high, 1000):
            end = min(start + 1000, high)
            for team in (0, 1):
                previous = positions[start - 10:end - 10, team]
                current = positions[start:end, team]
                distances = np.linalg.norm(current[:, :, None, :] - previous[:, None, :, :], axis=-1)
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore', RuntimeWarning)
                    forward = np.nanmin(distances, axis=2)
                    backward = np.nanmin(distances, axis=1)
                    values = np.nanquantile(np.concatenate([forward, backward], axis=1),
                                            [.25, .5, .75], axis=1).T
                motion[start:end, team * 3:team * 3 + 3] = np.nan_to_num(values) / 5

    base = np.clip(np.concatenate([quartiles, counts, motion], axis=1).astype(np.float32), -5, 5)
    result = [base]
    # Preserve the trained column order: current, three past means, three future means.
    for direction in ('past', 'future'):
        for width in (10, 30, 100):
            average = np.zeros_like(base)
            for low, high in input_blocks(data):
                cumulative = np.concatenate([np.zeros((1, 20)),
                    np.cumsum(base[low:high], axis=0, dtype=np.float64)])
                if direction == 'past':
                    end = np.arange(1, high - low + 1)
                    start = np.maximum(0, end - width)
                else:
                    start = np.arange(high - low)
                    end = np.minimum(start + width, high - low)
                average[low:high] = (cumulative[end] - cumulative[start]) / (end - start)[:, None]
            result.append(average)
    features = np.concatenate(result, axis=1).astype(np.float32)
    assert features.shape == (length, 140) and np.isfinite(features).all()
    return features
